In [1]:
from typing import Dict
from unittest import result

from dotenv import load_dotenv
from pyasn1.debug import scope

load_dotenv()

True

In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

def check_weather(city:str)->str:
    """Get the weather for a city."""
    forcasts={
        "goa":"sunny 32c,light humidity-good beach weather",
        "Hyd":"party cloud 29c"
    }
    return forcasts.get(city.strip().lower(),f"weather data not available for {city}")

agent=create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather]
)
print("agent ready with 1 tool")

agent ready with 1 tool


In [2]:
import os

# Set ONE of these depending on the provider you're using.
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

MODEL_NAME = "groq:llama-3.3-70b-versatile"

In [5]:
stream=agent.stream_events(
    {"messages":[{"role":"user","content":"what is the weather Today in goa?"}]},
    version="v3"
)
for message in stream.messages:
    for delta in message.text:
        print(delta,end='',flush=True)

E:\vs-langchain-agents\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
E:\vs-langchain-agents\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


The weather in Goa today is sunny with a temperature of 32 degrees Celsius and low humidity, making it ideal for beach activities.

In [6]:
stream=agent.stream_events(
    {"messages":[{"role":"user","content":"what is the weather Goa,and should i pack umbrella?"}]},
    version="v3"
)
for message in stream.messages:
    for delta in message.reasoning:
        print(f"[thinking] {delta},end="",flush=True)")

        for delta in message.text:
            print(delta,end='',flush=True)

In [7]:
from typing import Dict, Any
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
def search_flights(origin: str, destination: str) -> dict:
    """Search for a one-way flight and its price."""
    return {
        "origin": origin,
        "destination": destination,
        "one_way_price_inr": 9000,
        "duration": "2h 10m",
    }


agent = create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather, search_flights],
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Find me a flight from Hyderabad to Goa, and tell me the price."}]},
    version="v3",
)

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"[model is writing a tool call] {chunk}")

    finalized = message.tool_calls.get()   # the completed call, once the model is done writing it
    if finalized:
        print("[finalized tool call]", finalized)

    for delta in message.text:
        print(delta, end="", flush=True)

[finalized tool call] [{'type': 'tool_call', 'name': 'search_flights', 'args': {'destination': 'Goa', 'origin': 'Hyderabad'}, 'id': 'kqan43h98'}]
The price of a one-way flight from Hyderabad to Goa is ₹9000, and the flight duration is 2 hours and 10 minutes.

In [8]:
def estimate_total_cost(flight_one_way_price_inr:float,nights:int,hotel_per_night_inr:float=3000)->dict:
    """total estimate total trip cost round-trip flight+hotel for the given nights"""
    flight_total=flight_one_way_price_inr*2
    hotel_total=hotel_per_night_inr * nights
    total=flight_total+hotel_total
    return {
        "flight_total_inr":flight_total,
        "hotel_total_inr":hotel_total,
        "total_inr":total,
    }
SYSTEM_PROMPT="""You are a careful trip-planning assistant.

For every request, follow this loop and show your work in the chat:
1. PLAN: Write a short numbered plan (2-4 steps) before calling any tool.
2. EXECUTE: Call tools one at a time to carry out the plan.
3. CHECK: Once you know the total cost, compare it against the user's stated budget.
4. RE-PLAN IF NEEDED: If the total is over budget, say "Revising the plan:" followed by
   what you're changing, then execute the revised plan.
5. RECOMMEND: Give one final, clear recommendation.

The flight price you get back is fixed - if the trip is over budget, reduce the number
of nights instead, and call estimate_total_cost again with the new number.
"""
agent=create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_weather, search_flights,estimate_total_cost],
    system_prompt=SYSTEM_PROMPT,
)
print("agent ready with 3 tools and plan-execute-replan system_prompt")

agent ready with 3 tools and plan-execute-replan system_prompt


In [11]:
user_request = (
    "Plan a 5-night trip from Goa to Hyderabad. "
    "My total budget is INR 25,000. "
    "Also tell me what the weather will be like."
)

stream = agent.stream_events(
    {
        "messages": [
            {
                "role": "user",
                "content": user_request,
            }
        ]
    },
    version="v3",
)

for name, item in stream.interleave("messages", "tool_calls"):

    if name == "messages":

        # Stream text
        for chunk in getattr(item, "text", []):
            print(chunk, end="", flush=True)

        # Stream reasoning (if supported)
        for delta in getattr(item, "reasoning", []):
            print(f"[thinking] {delta}", end="", flush=True)

        # Stream model tool call writing
        for chunk in getattr(item, "tool_calls", []):
            print(f"[model is writing a tool call] {chunk}",
                  end="", flush=True)

    elif name == "tool_calls":
        print(
            f"\n[tool run] {item.tool_name}({item.input}) -> {item.output}"
        )

print("\n\n----------------------------- run finished -----------------------------")

Here's a short plan to help you with your trip:
1. Search for a one-way flight from Goa to Hyderabad to get the flight price.
2. Estimate the total cost of the trip, including the flight and hotel stay for 5 nights.
3. Check the weather in Hyderabad during the trip.
4. Compare the total cost with the budget and revise the plan if necessary.


[tool run] search_flights({'destination': 'Hyderabad', 'origin': 'Goa'}) -> None

[tool run] estimate_total_cost({'flight_one_way_price_inr': 9000, 'hotel_per_night_inr': 3000, 'nights': 5}) -> None

[tool run] check_weather({'city': 'Hyderabad'}) -> None
The total cost for the 5-night trip from Goa to Hyderabad is INR 33,000, which exceeds your budget of INR 25,000.

Revising the plan: To reduce the cost, I will decrease the number of nights from 5 to 3.

[tool run] estimate_total_cost({'flight_one_way_price_inr': 9000, 'hotel_per_night_inr': 3000, 'nights': 3}) -> None
Here's a 3-night trip plan from Goa to Hyderabad:

1. Travel from Goa to Hyde

In [16]:
from langgraph.config import get_stream_writer
from langgraph.stream import ProtocolEvent, StreamChannel, StreamTransformer, stream_channel


def check_wheather_verbose(city:str)->str:
    """Get the current weather forcast for a city ,narating its own progres"""
    writer=get_stream_writer()
    writer(f"Looking up weather for {city}")

    forcasts={
        "goa":"sunny 32c,lightly humdity good beach weather",
        "Hyd":"partly cloud 29c"
    }
    result=forcasts.get(city.strip().lower(),f"wheather data not available for {city}")
    writer(f'Done looking up weather for {city}')
    return result

from langgraph.stream import (
    StreamTransformer,
    StreamChannel,
    ProtocolEvent,
)

class ToolProgressTransformer(StreamTransformer):
    required_stream_modes = ("custom",)

    def __init__(self, scope: tuple[str, ...] = ()) -> None:
        super().__init__(scope)
        self.log = StreamChannel()

    def init(self):
        return {"custom": self.log}

    def process(self, event: ProtocolEvent) -> bool:
        if event.get("method") == "custom":
            self.log.push(event["params"]["data"])

        return True

In [17]:
agent=create_agent(
    model=init_chat_model(MODEL_NAME),
    tools=[check_wheather_verbose],
)
print("agent ready with tool narate its own progress")

agent ready with tool narate its own progress


In [18]:
stream=agent.stream_events(
    {"messages":[{"role":"user","content":"what is the wheather in goa?"}] },
    version="v3",
    transformers=[ToolProgressTransformer],
)
for name, item in stream.interleave("messages", "custom"):
    if name == "messages":
        for delta in item.text:
            print(delta, end="", flush=True)
    elif name == "custom":
        print(f"\n[tool progress {item}")


[tool progress Looking up weather for Goa

[tool progress Done looking up weather for Goa
The current weather in Goa is sunny with a temperature of 32 degrees Celsius and light humidity, making it good beach weather.